# Level 8 sub-basin analysis

In [ ]:
import pandas as pd
import geopandas as gpd
import os
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
def sjoin_summarize(points_gdf, poly_gdf, poly_field):
    
    joined_gdf = gpd.sjoin(points_gdf, poly_gdf, predicate='within', how='inner')
    return joined_gdf[['area_ha', poly_field]].groupby(poly_field).agg(['sum', 'count', 'median', 'mean'])['area_ha']

def sjoin_summarize_nogroup(points_gdf, poly_gdf):
    joined_gdf = gpd.sjoin(points_gdf, poly_gdf, predicate='within', how='inner')
    return joined_gdf[['area_ha']].agg(['sum', 'count', 'median', 'mean'])
def read_process_csv_to_gdf(csv):
    temp_df = pd.read_csv(csv)
    temp_gdf = gpd.GeoDataFrame(
        temp_df, geometry=gpd.points_from_xy(temp_df.longitude, temp_df.latitude),
        crs='EPSG:4326'
    )
    return temp_gdf

In [ ]:

reservoir_gdf = gpd.read_file('../clean_summarize/out_polys/sentinel_2021_v7_aea_cleaned_points.gpkg').to_crs('EPSG:4326')
dilate_gdf = read_process_csv_to_gdf('../clean_summarize/out_erode_dilate/sentinel_2021_v7_aea_dilate_cleaned_clip_id.csv')
erode_gdf = read_process_csv_to_gdf('../clean_summarize/out_erode_dilate/sentinel_2021_v7_aea_erode_cleaned_clip_id.csv')

In [ ]:
biome_gdf = gpd.read_file('./data/lm_bioma_250.shp').to_crs('EPSG:4326')
basin_gdf = gpd.read_file('./data/level8_basins_strahler_brazil.gpkg').to_crs('EPSG:4326')

In [ ]:
basin_gdf['nustrahler'] = basin_gdf['nustrahler'].fillna(0)

In [ ]:
basin_gdf['HYBAS_ID'] = basin_gdf['nunivotto8']
basin_gdf['ORDER'] = basin_gdf['nustrahler'].fillna(0).astype(int)

In [ ]:
# If necessary to clip
# state_gdf = gpd.read_file('./data/Brazilian_States.shp').to_crs('EPSG:4326')
# basin_gdf = basin_gdf.clip(state_gdf)
# basin_gdf.to_file('./data/level8_basins_strahler.gpkg')

In [ ]:
ws_res_gdf = sjoin_summarize(reservoir_gdf, basin_gdf, 'HYBAS_ID')
dilate_joined = sjoin_summarize(dilate_gdf, basin_gdf, 'HYBAS_ID')
erode_joined = sjoin_summarize(erode_gdf, basin_gdf, 'HYBAS_ID')

In [ ]:
joined_gdf = basin_gdf.set_index('HYBAS_ID').join(ws_res_gdf).fillna(0)
joined_gdf = joined_gdf.join(dilate_joined, rsuffix='_dilate').fillna(0)
joined_gdf = joined_gdf.join(erode_joined, rsuffix='_erode').fillna(0)

In [ ]:
joined_gdf.loc[joined_gdf['ORDER']==0, 'ORDER'] = 1

In [ ]:
joined_gdf['area_km2'] = joined_gdf.to_crs('ESRI:102033').geometry.area/(1000*1000)

In [ ]:
joined_gdf['count_density'] = joined_gdf['count']/joined_gdf['area_km2']
joined_gdf['area_density'] = joined_gdf['sum']/joined_gdf['area_km2']
joined_gdf['count_dilate_density'] = joined_gdf['count_dilate']/joined_gdf['area_km2']
joined_gdf['area_dilate_density'] = joined_gdf['sum_dilate']/joined_gdf['area_km2']
joined_gdf['count_erode_density'] = joined_gdf['count_erode']/joined_gdf['area_km2']
joined_gdf['area_erode_density'] = joined_gdf['sum_erode']/joined_gdf['area_km2']
joined_gdf['zero_count'] = (joined_gdf['count'] == 0)

In [ ]:
# Print various stats
# joined_gdf.groupby('ORDER').size()
# joined_gdf.groupby('ORDER')['zero_count'].mean()
# joined_gdf.groupby('ORDER')[['sum','count', 'mean']].mean()
# joined_gdf.groupby('ORDER')[['sum','count']].mean()
# joined_gdf.groupby('ORDER')[['sum','count', 'area_km2']].sum()
joined_gdf.groupby('ORDER')[['count_density','count_dilate_density', 'count_erode_density']].mean()

# ANA and GDW analysis

In [ ]:
ana_gdf = gpd.read_file('../compare_previous_results/data/ana/Massas_d_Agua.shp')
ana_gdf['geometry'] = ana_gdf.geometry.centroid
ana_gdf = ana_gdf.to_crs("EPSG:4326")

In [ ]:
ana_gdf['area_ha'] = 1
sjoin_summarize(ana_gdf.loc[ana_gdf['usoprinc']=='Hidrelétrica'], basin_gdf, 'ORDER')

In [ ]:
sjoin_summarize(ana_gdf.loc[ana_gdf['detipomass']=='Artificial'], basin_gdf, 'ORDER')

In [ ]:
gdw = gpd.read_file('../compare_previous_results/data/gdw/GDW_v1_0_shp/GDW_barriers_v1_0.shp').to_crs('EPSG:4326')

In [ ]:
gdw['area_ha'] = 1
sjoin_summarize(gdw, basin_gdf, 'HYBAS_ID')

In [ ]:
sjoin_summarize(gdw.loc[~gdw['USE_ELEC'].isna()], basin_gdf, 'ORDER').sum()

# Hydroregion analysis

In [ ]:
hydroregion = gpd.read_file('./data/macro_RH.shp').to_crs('EPSG:4326')
hydroregion_joined = hydroregion.sjoin(joined_gdf)

In [ ]:

hydroregion_grouped = hydroregion_joined.groupby(['nm_macroRH', 'ORDER'])['count'].sum()
for hname in hydroregion['nm_macroRH']:
    temp_df = hydroregion_grouped[hname]
    print(hname)
    print(temp_df[:3].sum()/temp_df.sum())
    print(temp_df[:3].sum()/1000)

In [ ]:
hydroregion_grouped = hydroregion_joined.groupby(['nm_macroRH', 'ORDER'])['count'].sum()
for hname in hydroregion['nm_macroRH']:
    temp_df = hydroregion_grouped[hname]
    print(hname)
    print(temp_df[-3:].sum()/temp_df.sum())
    print(temp_df[-3:].sum()/1000)

# Plots

In [ ]:
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

In [ ]:
# # Without secondary colormap
# # side_by_side_cmap=truncate_colormap(plt.cm.OrRd, 0.05, 0.9)  #'GnBu'
# area_cmap=truncate_colormap(plt.cm.Blues,0.1,0.9)
# count_cmap=truncate_colormap(plt.cm.Purples,0.1,0.9)
# # side_by_side_cmap=plt.cm.GnBu #'gist_yarg' # for grayscale

# COUNT_VMAX = 1.5
# axes_height_ratios=[1, 0.08, 1, 0.08]
# fig, axs = plt.subplots(4,1, figsize=(2.75,6.5), constrained_layout=True,
#                     gridspec_kw={"height_ratios":axes_height_ratios})

# # Count
# ax=axs[0]
# joined_gdf.clip([-74, -33, -33, 4]).plot(column='count_density',
#             ax=ax, cmap=count_cmap, legend=False,
#             edgecolor='grey', linewidth=0.005, vmax=COUNT_VMAX)
# biome_gdf.clip([-74, -33, -33, 4]).boundary.plot(
#             ax=ax,
#             edgecolor='grey', linewidth=0.1, alpha=0.5)
# ax.set_xlabel('Lon (deg)')
# ax.set_ylabel('Lat (deg)')
# # ax.legend()

# ax = axs[1]
# cmap = count_cmap # Choose the colormap
# norm = Normalize(vmin=0, vmax=COUNT_VMAX)
# sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])  # Required for the ScalarMappable
# cb = fig.colorbar(sm, cax=ax, orientation='horizontal',
#                 ticks=np.arange(0, COUNT_VMAX+0.01, 0.5))
# ax.set_title('Reservoirs per km$^2$')
# cb.ax.set_xticklabels([0, 0.5, 1.0,"$\geq$1.5"])
# # cb.ax.set_xticklabels([0, 0.2, 0.4, 0.6, 0.8, "$\geq$1.0"])

# # Arenaea
# ax=axs[2]
# AREA_VMAX = 2.0
# joined_gdf.clip([-74, -33, -33, 4]).plot(column='area_density',
#             ax=ax, cmap=area_cmap, legend=False,
#             edgecolor='grey', linewidth=0.005, vmax=AREA_VMAX)
# biome_gdf.clip([-74, -33, -33, 4]).boundary.plot(
#             ax=ax,
#             edgecolor='grey', linewidth=0.1, alpha=0.5)
# ax.set_xlabel('Lon (deg)')
# ax.set_ylabel('Lat (deg)')

# ax=axs[3]
# cmap = area_cmap # Choose the colormap
# norm = Normalize(vmin=0, vmax=AREA_VMAX)
# sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])  # Required for the ScalarMappable
# cb = fig.colorbar(sm, cax=ax, orientation='horizontal',
#                 ticks=np.arange(0, AREA_VMAX+0.01, 0.5))
# ax.set_title('Reservoir area (ha) per km$^2$')
# # cb.ax.set_xticklabels([0, 0.2, 0.4, 0.6, 0.8, "$\geq$1.0"])
# cb.ax.set_xticklabels([0, 0.5, 1.0,1.5,"$\geq$2.0"])

# axs[0].annotate(
#         "A",
#         xy=(0, 1), xycoords='axes fraction',
#         xytext=(0.25, -1.3), textcoords='offset fontsize',
#         fontsize=12, verticalalignment='bottom', fontfamily='serif')
# axs[2].annotate(
#         "B",
#         xy=(0, 1), xycoords='axes fraction',
#         xytext=(0.25, -1.3), textcoords='offset fontsize',
#         fontsize=12, verticalalignment='bottom', fontfamily='serif')
    
# plt.show()
# # plt.savefig('/home/ksolvik/research/reservoirs/figs/ch0/cafi_v1.tif', dpi=400,
# #             pil_kwargs={'compress':'lzw'},
# #             bbox_inches='tight')

In [ ]:
hex_codes = sns.color_palette("rocket_r", 4).as_hex()
edge_colormap = {}
for i, hex in enumerate(hex_codes):
    edge_colormap[i+1] = hex

In [ ]:
joined_gdf['zero_count'].mean()

In [ ]:
joined_gdf.groupby('ORDER')['zero_count'].mean()

In [ ]:
joined_gdf.groupby('ORDER')['count'].sum()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm, Normalize

from mpl_toolkits.axes_grid1 import make_axes_locatable

# Define colormaps
count_cmap = plt.cm.GnBu
order_cmap = plt.cm.get_cmap('flare', 10)

# Figure setup
fig = plt.figure(figsize=(10, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.5], wspace=0.06)

# Top-left: Sub-basin order map
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_xlim(-75, -33)
ax0.set_ylim(-35, 6)
joined_gdf.plot(column='ORDER', categorical=True, ax=ax0, cmap='flare', legend=False, linewidth=0.00)
ax0.set_xlabel('Lon (deg)')
ax0.set_ylabel('Lat (deg)')
# ax0.set_title('Sub-basin Max Strahler Order', fontsize=12)
divider0 = make_axes_locatable(ax0)
cax0 = divider0.append_axes("right", size="5%", pad=0.01)
bounds = np.arange(1, 12)
n_colors = len(bounds) - 1
norm0 = mcolors.BoundaryNorm(bounds, n_colors)
sm0 = plt.cm.ScalarMappable(cmap=order_cmap, norm=norm0)
sm0.set_array([])
cb0 = fig.colorbar(sm0, cax=cax0, orientation='vertical')
cb0.set_ticks(bounds[:-1] + 0.5)
cb0.set_ticklabels([str(i) for i in range(1, 11)])
cb0.ax.set_ylabel('Sub-basin Order', fontsize=12)

# Top-right: Count map
ax1 = fig.add_subplot(gs[0, 1])
COUNT_VMAX = 50
joined_gdf.plot(column='count', ax=ax1, cmap=count_cmap, legend=False, edgecolor='grey', linewidth=0.0, vmax=COUNT_VMAX)
# biome_gdf.boundary.plot(ax=ax1, edgecolor='grey', linewidth=0.1, alpha=0.5)
ax1.set_xlim(-75, -33)
ax1.set_ylim(-35, 6)
ax1.set_xlabel('Lon (deg)')
# ax1.set_ylabel('Lat (deg)')
# ax1.set_title('Reservoirs per sub-basin', fontsize=12)
divider1 = make_axes_locatable(ax1)

cax1 = divider1.append_axes("right", size="5%", pad=0.01)
norm1 = Normalize(vmin=0, vmax=COUNT_VMAX)
sm1 = plt.cm.ScalarMappable(cmap=count_cmap, norm=norm1)
sm1.set_array([])
cb1 = fig.colorbar(sm1, cax=cax1, orientation='vertical')
cb1.set_ticks(np.arange(0, COUNT_VMAX + 0.01, 10))
cb1.ax.set_yticklabels([0, 10, 20, 30, 40, '≥50'], rotation=90, va='center')
cb1.ax.set_ylabel('Reservoirs per sub-basin', fontsize=12)

# Bottom: Barplot
ax2 = fig.add_subplot(gs[1, :])
sns.barplot(data=joined_gdf, x='ORDER', y='count_density', hue='ORDER', palette='flare', legend=False, errorbar=None, ax=ax2)
grouped = joined_gdf.groupby('ORDER')['count_density'].mean()
lower = grouped - joined_gdf.groupby('ORDER')['count_erode_density'].mean()
upper = joined_gdf.groupby('ORDER')['count_dilate_density'].mean() - grouped
yerr = np.array([lower.values, upper.values])
ax2.errorbar(x=range(len(grouped)), y=grouped, yerr=yerr, fmt='none', c='black', capsize=3, linewidth=1)
ax2.set_xlabel('Sub-basin Order', fontsize=12)
ax2.set_ylabel('Mean # Reservoirs\nper km²', fontsize=12)
ax2.set_ylim(0, 0.2)
ax2.set_xticklabels(np.arange(1, 11))
ax2.set_yticks(np.arange(0, 0.201, 0.05))
# ax2.set_yticklabels(np.arange(0, 0.201, 0.05))

# Annotations
ax0.annotate("A", xy=(0.02, 0.94), xycoords='axes fraction', fontsize=14, fontfamily='serif', fontweight='bold')
ax1.annotate("B", xy=(0.02, 0.94), xycoords='axes fraction', fontsize=14, fontfamily='serif', fontweight='bold')
ax2.annotate("C", xy=(0.01, 0.91), xycoords='axes fraction', fontsize=14, fontfamily='serif', fontweight='bold')

# fig.tight_layout()
plt.savefig('/home/ksolvik/research/reservoirs/figs/ch0/combined_basin_fig.tif', dpi=400, pil_kwargs={'compress':'lzw'}, bbox_inches='tight')
# plt.show()